In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
os.makedirs(SAVE_DIR, exist_ok=True)
CKPT = f'{SAVE_DIR}/teacher_best.pt'

Mounted at /content/drive


# QST 1 : Entrainement du teacher
sur CIFAR10, avec un split train/val/test, et compter les paramètres.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision, torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"

# Données : CIFAR10 redimensionné à 224 (ou 64/128 pour aller plus vite)
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean,std)])

full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)

# Split 45000 / 5000 (reproductible)
g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_set = torch.utils.data.Subset(full_train, idx_train.indices)
val_set   = torch.utils.data.Subset(full_val,   idx_val.indices)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

# Modèle teacher
teacher = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher = teacher.to(device)

# Nombre de paramètres
total = sum(p.numel() for p in teacher.parameters())
trainable = sum(p.numel() for p in teacher.parameters() if p.requires_grad)
print(f"Teacher ResNet50 : {total:,} paramètres ({trainable:,} entraînables)")

# Entraînement / évaluation
def evaluate(model, loader):
    model.eval(); correct = total_n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); total_n += y.size(0)
    return loss_sum/total_n, correct/total_n

def train(model, epochs=5, lr=1e-3):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(model, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(model.state_dict(), "teacher_best.pt")
    return hist

hist = train(teacher, epochs=5)

100%|██████████| 170M/170M [20:24<00:00, 139kB/s]


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 160MB/s]


Teacher ResNet50 : 23,528,522 paramètres (23,528,522 entraînables)
Epoch 1/5 | val loss 0.2191 | val acc 0.9294
Epoch 2/5 | val loss 0.1469 | val acc 0.9480
Epoch 3/5 | val loss 0.1286 | val acc 0.9558
Epoch 4/5 | val loss 0.1181 | val acc 0.9608
Epoch 5/5 | val loss 0.1120 | val acc 0.9624


NameError: name 'model' is not defined

In [ ]:
torch.save(teacher.state_dict(), CKPT)
print(os.path.exists(CKPT), os.path.getsize(CKPT)/1e6, "Mo")

True 94.433693 Mo


In [ ]:
tl, ta = evaluate(teacher, test_loader)
print(f"TEST : loss {tl:.4f} | acc {ta:.4f}")

TEST : loss 0.1284 | acc 0.9553


# Strategie 1
## QST 2 : Entrainement de l'éleve
(pré-entraîné sur ImageNet) sur le jeu de données CIFAR10, tout en procédant à son évaluation.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
CKPT_S1 = f'{SAVE_DIR}/student_pre_best.pt'

# Même fonction que pour le teacher, avec le chemin de sauvegarde en paramètre
def train(model, ckpt, epochs=5, lr=1e-3):
    opt = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = nn.functional.cross_entropy(model(x), y)
            loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(model, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(model.state_dict(), ckpt)
    return hist

# Student pré-entraîné ImageNet
student_pre = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_pre.fc = nn.Linear(student_pre.fc.in_features, 10)
student_pre = student_pre.to(device)

total = sum(p.numel() for p in student_pre.parameters())
trainable = sum(p.numel() for p in student_pre.parameters() if p.requires_grad)
print(f"Student ResNet18 : {total:,} paramètres ({trainable:,} entraînables)")

hist_s1 = train(student_pre, CKPT_S1, epochs=5)

# Test avec le meilleur checkpoint
student_pre.load_state_dict(torch.load(CKPT_S1, map_location=device))
sl, sa = evaluate(student_pre, test_loader)
print(f"TEST student : loss {sl:.4f} | acc {sa:.4f}")

# Comparaison avec le teacher
tp = sum(p.numel() for p in teacher.parameters())
print(f"Ratio paramètres teacher/student : {tp/total:.2f}x")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


Student ResNet18 : 11,181,642 paramètres (11,181,642 entraînables)
Epoch 1/5 | val loss 0.2941 | val acc 0.9058
Epoch 2/5 | val loss 0.2224 | val acc 0.9240
Epoch 3/5 | val loss 0.1915 | val acc 0.9354
Epoch 4/5 | val loss 0.1781 | val acc 0.9400
Epoch 5/5 | val loss 0.1770 | val acc 0.9386
TEST student : loss 0.1913 | acc 0.9347
Ratio paramètres teacher/student : 2.10x


## QST 3 : Figer les paramètres du réseau teacher et entrainer l'élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances au niveau des scores.

In [ ]:
import copy

CKPT_S3 = f'{SAVE_DIR}/student_kd_scores_best.pt'

# Teacher gelé
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

def kd_scores_loss(s_logits, t_logits, y, alpha=0.5):
    ce  = nn.functional.cross_entropy(s_logits, y)
    mse = nn.functional.mse_loss(s_logits, t_logits)
    return (1 - alpha) * ce + alpha * mse, ce.item(), mse.item()

def train_kd(student, ckpt, epochs=5, lr=1e-3, alpha=0.5):
    opt = optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            loss, ce, mse = kd_scores_loss(s_logits, t_logits, y, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce:.4f} | MSE {mse:.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

# Nouveau student pré-entraîné ImageNet (mêmes conditions de départ qu'à la question 2)
student_kd = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_kd.fc = nn.Linear(student_kd.fc.in_features, 10)
student_kd = student_kd.to(device)

hist_s3 = train_kd(student_kd, CKPT_S3, epochs=5, alpha=0.5)

student_kd.load_state_dict(torch.load(CKPT_S3, map_location=device))
kl, ka = evaluate(student_kd, test_loader)
print(f"TEST student KD scores : loss {kl:.4f} | acc {ka:.4f}")
print(f"Gain vs student sans KD : {100*(ka - sa):+.2f} points")

Epoch 1/5 | CE 0.1624 | MSE 1.1059 | val loss 0.2167 | val acc 0.9206
Epoch 2/5 | CE 0.1140 | MSE 1.2896 | val loss 0.1763 | val acc 0.9380
Epoch 3/5 | CE 0.1735 | MSE 1.0395 | val loss 0.1611 | val acc 0.9440
Epoch 4/5 | CE 0.1487 | MSE 0.7960 | val loss 0.1548 | val acc 0.9470
Epoch 5/5 | CE 0.1066 | MSE 0.6679 | val loss 0.1533 | val acc 0.9474
TEST student KD scores : loss 0.1659 | acc 0.9438
Gain vs student sans KD : +0.91 points


## QST 4 : Figer les paramètres du réseau teacher et entraîner le réseau élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances, tant au niveau des scores qu'au niveau des cartes de caractéristiques.

In [ ]:
CKPT_S4 = f'{SAVE_DIR}/student_kd_feat_best.pt'

def forward_feats(model, x):
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    x = model.layer1(x); x = model.layer2(x); x = model.layer3(x)
    f = model.layer4(x)                         # feature map finale
    logits = model.fc(torch.flatten(model.avgpool(f), 1))
    return f, logits

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def train_kd_feat(student, projector, ckpt, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce   = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

student_kd2 = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
student_kd2.fc = nn.Linear(student_kd2.fc.in_features, 10)
student_kd2 = student_kd2.to(device)
projector = Projector(512, 2048).to(device)

hist_s4 = train_kd_feat(student_kd2, projector, CKPT_S4, epochs=5, alpha=0.5, beta=1.0)

student_kd2.load_state_dict(torch.load(CKPT_S4, map_location=device))
fl, fa = evaluate(student_kd2, test_loader)
print(f"TEST student KD scores+features : loss {fl:.4f} | acc {fa:.4f}")
print(f"Gain vs sans KD : {100*(fa - sa):+.2f} pts | vs KD scores : {100*(fa - ka):+.2f} pts")

Epoch 1/5 | CE 0.3890 | MSE logits 1.3062 | MSE feat 1.6465 | val loss 0.2204 | val acc 0.9212
Epoch 2/5 | CE 0.2081 | MSE logits 0.9730 | MSE feat 1.4983 | val loss 0.1793 | val acc 0.9372
Epoch 3/5 | CE 0.1932 | MSE logits 0.9236 | MSE feat 1.3950 | val loss 0.1653 | val acc 0.9430
Epoch 4/5 | CE 0.1262 | MSE logits 0.7466 | MSE feat 1.3099 | val loss 0.1585 | val acc 0.9464
Epoch 5/5 | CE 0.1802 | MSE logits 1.0378 | MSE feat 1.3273 | val loss 0.1569 | val acc 0.9454
TEST student KD scores+features : loss 0.1686 | acc 0.9418
Gain vs sans KD : +0.71 pts | vs KD scores : -0.20 pts


# Stratégie 2 : Distillation de connaissances utilisant un modèle élève non pré-entraîné
## QST 5 : Entraîner le réseau élève (non pré-entraîné sur ImageNet) à partir de zéro
sur le jeu de données CIFAR10, tout en procédant à son évaluation. Veiller à une répartition appropriée des données pour l'entraînement, la validation et le test.

In [ ]:
CKPT_S5 = f'{SAVE_DIR}/student_scratch_best.pt'
EPOCHS_SCRATCH = 15
LR_SCRATCH = 0.05

student_scratch = resnet18(weights=None)
student_scratch.fc = nn.Linear(student_scratch.fc.in_features, 10)
student_scratch = student_scratch.to(device)

total = sum(p.numel() for p in student_scratch.parameters())
print(f"Student ResNet18 from scratch : {total:,} paramètres")

hist_s5 = train(student_scratch, CKPT_S5, epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH)

student_scratch.load_state_dict(torch.load(CKPT_S5, map_location=device))
cl, ca = evaluate(student_scratch, test_loader)
print(f"TEST student scratch : loss {cl:.4f} | acc {ca:.4f}")

Student ResNet18 from scratch : 11,181,642 paramètres
Epoch 1/15 | val loss 1.2883 | val acc 0.5328
Epoch 2/15 | val loss 1.0304 | val acc 0.6416
Epoch 3/15 | val loss 0.6604 | val acc 0.7668
Epoch 4/15 | val loss 0.9025 | val acc 0.6834
Epoch 5/15 | val loss 0.7355 | val acc 0.7502
Epoch 6/15 | val loss 0.6205 | val acc 0.7896
Epoch 7/15 | val loss 0.5527 | val acc 0.8166
Epoch 8/15 | val loss 0.4991 | val acc 0.8320
Epoch 9/15 | val loss 0.6867 | val acc 0.7912
Epoch 10/15 | val loss 0.4357 | val acc 0.8590
Epoch 11/15 | val loss 0.3870 | val acc 0.8746
Epoch 12/15 | val loss 0.3644 | val acc 0.8916
Epoch 13/15 | val loss 0.3544 | val acc 0.8942
Epoch 14/15 | val loss 0.3463 | val acc 0.8968
Epoch 15/15 | val loss 0.3468 | val acc 0.8966
TEST student scratch : loss 0.3656 | acc 0.8871


## QST 6 : Figer les paramètres du réseau teacher et entraîner le réseau élève
sous la supervision de l'enseignant, en utilisant la distillation de connaissances au niveau des scores.

In [ ]:
CKPT_S6 = f'{SAVE_DIR}/student_scratch_kd_scores_best.pt'

student_scratch_kd = resnet18(weights=None)
student_scratch_kd.fc = nn.Linear(student_scratch_kd.fc.in_features, 10)
student_scratch_kd = student_scratch_kd.to(device)

hist_s6 = train_kd(student_scratch_kd, CKPT_S6,
                   epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5)

student_scratch_kd.load_state_dict(torch.load(CKPT_S6, map_location=device))
kl6, ka6 = evaluate(student_scratch_kd, test_loader)
print(f"TEST scratch KD scores : loss {kl6:.4f} | acc {ka6:.4f}")
print(f"Gain vs scratch sans KD : {100*(ka6 - ca):+.2f} points")

Epoch 1/15 | CE 1.0404 | MSE 4.2356 | val loss 1.4233 | val acc 0.5468
Epoch 2/15 | CE 0.6144 | MSE 2.8119 | val loss 0.7991 | val acc 0.7308
Epoch 3/15 | CE 0.4290 | MSE 2.2857 | val loss 0.8823 | val acc 0.7338
Epoch 4/15 | CE 0.4485 | MSE 2.0529 | val loss 0.5963 | val acc 0.8116
Epoch 5/15 | CE 0.4795 | MSE 2.0709 | val loss 0.5025 | val acc 0.8364
Epoch 6/15 | CE 0.2191 | MSE 1.5436 | val loss 0.4780 | val acc 0.8476
Epoch 7/15 | CE 0.2805 | MSE 1.3614 | val loss 0.4212 | val acc 0.8654
Epoch 8/15 | CE 0.1046 | MSE 1.0384 | val loss 0.3302 | val acc 0.8948
Epoch 9/15 | CE 0.0728 | MSE 1.1597 | val loss 0.3058 | val acc 0.8954
Epoch 10/15 | CE 0.1339 | MSE 0.8287 | val loss 0.2926 | val acc 0.9056
Epoch 11/15 | CE 0.1152 | MSE 0.7240 | val loss 0.2705 | val acc 0.9130
Epoch 12/15 | CE 0.0325 | MSE 0.5791 | val loss 0.2563 | val acc 0.9136
Epoch 13/15 | CE 0.0216 | MSE 0.6095 | val loss 0.2499 | val acc 0.9200
Epoch 14/15 | CE 0.0237 | MSE 0.6383 | val loss 0.2454 | val acc 0.9200
E

sauvegarde

In [ ]:
import os
print(os.path.exists(CKPT_S6), os.path.getsize(CKPT_S6)/1e6, "Mo")   # ~45 Mo attendu

True 44.808779 Mo


In [ ]:
import json
RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}

results.update({
    "teacher":              {"test_acc": ta},
    "student_pre":          {"test_acc": sa, "hist": hist_s1},
    "kd_scores":            {"test_acc": ka, "hist": hist_s3},
    "kd_scores_feat":       {"test_acc": fa, "hist": hist_s4},
    "scratch":              {"test_acc": ca, "hist": hist_s5},
    "scratch_kd_scores":    {"test_acc": ka6, "hist": hist_s6},
})
json.dump(results, open(RES, "w"), indent=2)

## QST 7 : Figer les paramètres du réseau teacher et entraîner le réseau élève sous la supervision du teacher
en utilisant la distillation de connaissances à la fois au niveau des scores et au niveau des cartes de caractéristiques (feature maps).

In [ ]:
import json

CKPT_S7 = f'{SAVE_DIR}/student_scratch_kd_feat_best.pt'
LAST_S7 = f'{SAVE_DIR}/student_scratch_kd_feat_last.pt'

def train_kd_feat_resume(student, projector, ckpt, last, epochs, lr, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist, start = 0, [], 0

    if os.path.exists(last):
        st = torch.load(last, map_location=device)
        student.load_state_dict(st['student']); projector.load_state_dict(st['proj'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best, hist, start = st['best'], st['hist'], st['epoch'] + 1
        print(f"Reprise à l'epoch {start+1}")

    for ep in range(start, epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce    = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
        torch.save({'epoch': ep, 'student': student.state_dict(), 'proj': projector.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'best': best, 'hist': hist}, last)
    return hist

student_scratch_kd2 = resnet18(weights=None)
student_scratch_kd2.fc = nn.Linear(student_scratch_kd2.fc.in_features, 10)
student_scratch_kd2 = student_scratch_kd2.to(device)
projector7 = Projector(512, 2048).to(device)

hist_s7 = train_kd_feat_resume(student_scratch_kd2, projector7, CKPT_S7, LAST_S7,
                               epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5, beta=1.0)

student_scratch_kd2.load_state_dict(torch.load(CKPT_S7, map_location=device))
fl7, fa7 = evaluate(student_scratch_kd2, test_loader)
print(f"TEST scratch KD scores+features : loss {fl7:.4f} | acc {fa7:.4f}")
print(f"Gain vs scratch sans KD : {100*(fa7 - ca):+.2f} pts | vs scratch KD scores : {100*(fa7 - ka6):+.2f} pts")

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
results["scratch_kd_scores_feat"] = {"test_acc": fa7, "hist": hist_s7}
json.dump(results, open(RES, "w"), indent=2)

Epoch 1/15 | CE 1.6354 | MSE logits 5.4033 | MSE feat 0.9359 | val loss 1.4064 | val acc 0.5392
Epoch 2/15 | CE 0.7532 | MSE logits 2.7808 | MSE feat 0.6098 | val loss 0.7589 | val acc 0.7534
Epoch 3/15 | CE 0.7622 | MSE logits 2.5631 | MSE feat 0.5668 | val loss 0.7960 | val acc 0.7538
Epoch 4/15 | CE 0.3419 | MSE logits 2.2663 | MSE feat 0.4946 | val loss 0.6013 | val acc 0.8170
Epoch 5/15 | CE 0.5470 | MSE logits 1.6613 | MSE feat 0.4486 | val loss 0.5556 | val acc 0.8244
Epoch 6/15 | CE 0.4999 | MSE logits 1.6066 | MSE feat 0.4282 | val loss 0.5501 | val acc 0.8340
Epoch 7/15 | CE 0.2307 | MSE logits 1.3779 | MSE feat 0.4324 | val loss 0.4194 | val acc 0.8698
Epoch 8/15 | CE 0.1880 | MSE logits 1.3843 | MSE feat 0.4060 | val loss 0.3728 | val acc 0.8856
Epoch 9/15 | CE 0.1188 | MSE logits 0.9647 | MSE feat 0.3943 | val loss 0.3152 | val acc 0.8956
Epoch 10/15 | CE 0.0333 | MSE logits 0.8555 | MSE feat 0.3752 | val loss 0.2742 | val acc 0.9096


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, torch, torch.nn as nn, torch.optim as optim
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader, random_split
from torchvision.models import resnet18, resnet50, ResNet18_Weights

device = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = '/content/drive/MyDrive/UNI/M2_VMI/LabKD1'
CKPT_S7  = f'{SAVE_DIR}/student_scratch_kd_feat_best.pt'
LAST_S7  = f'{SAVE_DIR}/student_scratch_kd_feat_last.pt'
EPOCHS_SCRATCH, LR_SCRATCH = 15, 0.05

print(device, os.path.exists(LAST_S7), os.path.exists(f'{SAVE_DIR}/teacher_best.pt'))

Mounted at /content/drive
cuda True True


In [ ]:
mean, std = (0.485,0.456,0.406), (0.229,0.224,0.225)
tf_train = T.Compose([T.Resize(224), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(mean,std)])
tf_eval  = T.Compose([T.Resize(224), T.ToTensor(), T.Normalize(mean,std)])

full_train = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_train)
full_val   = torchvision.datasets.CIFAR10("./data", train=True,  download=True, transform=tf_eval)
test_set   = torchvision.datasets.CIFAR10("./data", train=False, download=True, transform=tf_eval)

g = torch.Generator().manual_seed(42)
idx_train, idx_val = random_split(range(50000), [45000, 5000], generator=g)
train_set = torch.utils.data.Subset(full_train, idx_train.indices)
val_set   = torch.utils.data.Subset(full_val,   idx_val.indices)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

100%|██████████| 170M/170M [00:06<00:00, 28.1MB/s]


In [ ]:
def evaluate(model, loader):
    model.eval(); correct = total_n = 0; loss_sum = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += nn.functional.cross_entropy(out, y, reduction="sum").item()
            correct += (out.argmax(1) == y).sum().item(); total_n += y.size(0)
    return loss_sum/total_n, correct/total_n

def forward_feats(model, x):
    x = model.maxpool(model.relu(model.bn1(model.conv1(x))))
    x = model.layer1(x); x = model.layer2(x); x = model.layer3(x)
    f = model.layer4(x)                         # feature map finale
    logits = model.fc(torch.flatten(model.avgpool(f), 1))
    return f, logits

class Projector(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.proj = nn.Sequential(nn.Conv2d(c_in, c_out, 1, bias=False), nn.BatchNorm2d(c_out))
    def forward(self, x): return self.proj(x)

def train_kd_feat(student, projector, ckpt, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce   = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

def kd_scores_loss(s_logits, t_logits, y, alpha=0.5):
    ce  = nn.functional.cross_entropy(s_logits, y)
    mse = nn.functional.mse_loss(s_logits, t_logits)
    return (1 - alpha) * ce + alpha * mse, ce.item(), mse.item()

def train_kd(student, ckpt, epochs=5, lr=1e-3, alpha=0.5):
    opt = optim.SGD(student.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_logits = teacher(x)
            s_logits = student(x)
            loss, ce, mse = kd_scores_loss(s_logits, t_logits, y, alpha)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce:.4f} | MSE {mse:.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
    return hist

def train_kd_feat_resume(student, projector, ckpt, last, epochs, lr, alpha=0.5, beta=1.0):
    params = list(student.parameters()) + list(projector.parameters())
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    best, hist, start = 0, [], 0

    if os.path.exists(last):
        st = torch.load(last, map_location=device)
        student.load_state_dict(st['student']); projector.load_state_dict(st['proj'])
        opt.load_state_dict(st['opt']); sched.load_state_dict(st['sched'])
        best, hist, start = st['best'], st['hist'], st['epoch'] + 1
        print(f"Reprise à l'epoch {start+1}")

    for ep in range(start, epochs):
        student.train(); projector.train(); teacher.eval()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            ce    = nn.functional.cross_entropy(s_logits, y)
            mse_l = nn.functional.mse_loss(s_logits, t_logits)
            mse_f = nn.functional.mse_loss(projector(s_feat), t_feat)
            loss = (1 - alpha) * ce + alpha * mse_l + beta * mse_f
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader)
        hist.append((vl, va))
        print(f"Epoch {ep+1}/{epochs} | CE {ce.item():.4f} | MSE logits {mse_l.item():.4f} | MSE feat {mse_f.item():.4f} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va
            torch.save(student.state_dict(), ckpt)
        torch.save({'epoch': ep, 'student': student.state_dict(), 'proj': projector.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(),
                    'best': best, 'hist': hist}, last)
    return hist


In [ ]:
teacher = resnet50()
teacher.fc = nn.Linear(teacher.fc.in_features, 10)
teacher.load_state_dict(torch.load(f'{SAVE_DIR}/teacher_best.pt', map_location=device))
teacher = teacher.to(device).eval()
for p in teacher.parameters():
    p.requires_grad = False

In [ ]:
student_scratch_kd2 = resnet18(weights=None)
student_scratch_kd2.fc = nn.Linear(512, 10)
student_scratch_kd2 = student_scratch_kd2.to(device)
projector7 = Projector(512, 2048).to(device)

hist_s7 = train_kd_feat_resume(student_scratch_kd2, projector7, CKPT_S7, LAST_S7,
                               epochs=EPOCHS_SCRATCH, lr=LR_SCRATCH, alpha=0.5, beta=1.0)

Reprise à l'epoch 11
Epoch 11/15 | CE 0.0859 | MSE logits 0.7648 | MSE feat 0.3668 | val loss 0.2675 | val acc 0.9102
Epoch 12/15 | CE 0.0523 | MSE logits 0.6790 | MSE feat 0.4039 | val loss 0.2562 | val acc 0.9174
Epoch 13/15 | CE 0.1151 | MSE logits 0.7324 | MSE feat 0.3799 | val loss 0.2460 | val acc 0.9190
Epoch 14/15 | CE 0.0420 | MSE logits 0.6066 | MSE feat 0.3729 | val loss 0.2387 | val acc 0.9214
Epoch 15/15 | CE 0.0475 | MSE logits 0.6059 | MSE feat 0.3911 | val loss 0.2369 | val acc 0.9198


In [ ]:
student_scratch_kd2.load_state_dict(torch.load(CKPT_S7, map_location=device))
fl7, fa7 = evaluate(student_scratch_kd2, test_loader)
print(f"TEST scratch KD scores+features : loss {fl7:.4f} | acc {fa7:.4f}")

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}
results["scratch_kd_scores_feat"] = {"test_acc": fa7, "hist": hist_s7}
json.dump(results, open(RES, "w"), indent=2)

TEST scratch KD scores+features : loss 0.2424 | acc 0.9228


# Comparaisons
## Comparer Stratégie 1 et 2

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}


fallback = {"teacher": 0.9553, "student_pre": 0.9347, "kd_scores": 0.9438,
            "kd_scores_feat": 0.9418, "scratch": 0.8871,
            "scratch_kd_scores": None,          # à remplir avec ton résultat Q6
            "scratch_kd_scores_feat": 0.9228}
acc = {k: results.get(k, {}).get("test_acc", fallback[k]) for k in fallback}
assert acc["scratch_kd_scores"] is not None, "Renseigne la test acc de Q6 dans fallback"

rows = []
for strat, base, keys in [
    ("Stratégie 1 (pré-entraîné)", "student_pre", ["student_pre", "kd_scores", "kd_scores_feat"]),
    ("Stratégie 2 (from scratch)", "scratch",     ["scratch", "scratch_kd_scores", "scratch_kd_scores_feat"]),
]:
    for k, label in zip(keys, ["Sans KD", "KD scores", "KD scores + features"]):
        rows.append({"Stratégie": strat, "Méthode": label,
                     "Test acc (%)": 100*acc[k],
                     "Gain vs sans KD (pts)": 100*(acc[k] - acc[base]),
                     "Écart au teacher (pts)": 100*(acc[k] - acc["teacher"])})
df = pd.DataFrame(rows).round(2)
print(df.to_string(index=False))
df.to_csv(f'{SAVE_DIR}/comparaison_strategies.csv', index=False)


hist_keys = {"S1 sans KD": "student_pre", "S1 KD scores": "kd_scores", "S1 KD scores+feat": "kd_scores_feat",
             "S2 sans KD": "scratch", "S2 KD scores": "scratch_kd_scores", "S2 KD scores+feat": "scratch_kd_scores_feat"}
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for label, k in hist_keys.items():
    h = results.get(k, {}).get("hist")
    if h is None: continue
    a = ax[0] if label.startswith("S1") else ax[1]
    a.plot(range(1, len(h)+1), [x[1] for x in h], marker="o", label=label)
ax[0].set_title("Stratégie 1 : val acc"); ax[1].set_title("Stratégie 2 : val acc")
for a in ax: a.set_xlabel("Epoch"); a.set_ylabel("Val acc"); a.grid(alpha=.3); a.legend()
plt.tight_layout(); plt.savefig(f'{SAVE_DIR}/courbes_strategies.png', dpi=150); plt.show()

## Comparer différentes métriques de distance
entre l’enseignant et l’élève (autres que la MSE)

In [ ]:
import torch.nn.functional as F

def dist_logits(name, s, t, T=4.0):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    if name == "cosine": return (1 - F.cosine_similarity(s, t, dim=1)).mean()
    if name == "kl":
        return F.kl_div(F.log_softmax(s/T, 1), F.softmax(t/T, 1), reduction="batchmean") * T * T
    raise ValueError(name)

def dist_feat(name, s, t):
    if name == "mse":    return F.mse_loss(s, t)
    if name == "l1":     return F.l1_loss(s, t)
    if name == "huber":  return F.smooth_l1_loss(s, t)
    if name in ("cosine", "kl"):
        return (1 - F.cosine_similarity(s, t, dim=1)).mean()
    raise ValueError(name)

def run_kd(metric, use_feat, epochs=5, lr=1e-3, alpha=0.5, beta=1.0):
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    ckpt = f'{SAVE_DIR}/{tag}_best.pt'

    student = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    student.fc = nn.Linear(512, 10); student = student.to(device)
    proj = Projector(512, 2048).to(device) if use_feat else None
    params = list(student.parameters()) + (list(proj.parameters()) if use_feat else [])
    opt = optim.SGD(params, lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, epochs)

    best, hist = 0, []
    for ep in range(epochs):
        student.train(); teacher.eval()
        if proj: proj.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            with torch.no_grad():
                t_feat, t_logits = forward_feats(teacher, x)
            s_feat, s_logits = forward_feats(student, x)
            loss = (1 - alpha) * F.cross_entropy(s_logits, y) + alpha * dist_logits(metric, s_logits, t_logits)
            if use_feat:
                loss = loss + beta * dist_feat(metric, proj(s_feat), t_feat)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        vl, va = evaluate(student, val_loader); hist.append((vl, va))
        print(f"[{tag}] epoch {ep+1}/{epochs} | val loss {vl:.4f} | val acc {va:.4f}")
        if va > best:
            best = va; torch.save(student.state_dict(), ckpt)

    student.load_state_dict(torch.load(ckpt, map_location=device))
    tl, ta_ = evaluate(student, test_loader)
    print(f"[{tag}] TEST acc {ta_:.4f}")
    return tag, {"test_acc": ta_, "hist": hist}

RES = f'{SAVE_DIR}/results.json'
results = json.load(open(RES)) if os.path.exists(RES) else {}

# Série 1 : scores seuls. Série 2 : scores + features
configs = [(m, False) for m in ["l1", "huber", "cosine", "kl"]]


for metric, use_feat in configs:
    tag = f"q9_{metric}_{'feat' if use_feat else 'scores'}"
    if tag in results:
        print("déjà fait :", tag); continue
    tag, res = run_kd(metric, use_feat)
    results[tag] = res
    json.dump(results, open(RES, "w"), indent=2)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


[q9_l1_scores] epoch 1/5 | val loss 0.3322 | val acc 0.8890
[q9_l1_scores] epoch 2/5 | val loss 0.2329 | val acc 0.9136
[q9_l1_scores] epoch 3/5 | val loss 0.2030 | val acc 0.9258
[q9_l1_scores] epoch 4/5 | val loss 0.1918 | val acc 0.9302
[q9_l1_scores] epoch 5/5 | val loss 0.1888 | val acc 0.9314
[q9_l1_scores] TEST acc 0.9302
[q9_huber_scores] epoch 1/5 | val loss 0.3442 | val acc 0.8906
[q9_huber_scores] epoch 2/5 | val loss 0.2396 | val acc 0.9156
[q9_huber_scores] epoch 3/5 | val loss 0.2112 | val acc 0.9254
[q9_huber_scores] epoch 4/5 | val loss 0.1994 | val acc 0.9318
[q9_huber_scores] epoch 5/5 | val loss 0.1980 | val acc 0.9320
[q9_huber_scores] TEST acc 0.9275
[q9_cosine_scores] epoch 1/5 | val loss 0.3671 | val acc 0.8870
[q9_cosine_scores] epoch 2/5 | val loss 0.2610 | val acc 0.9116
[q9_cosine_scores] epoch 3/5 | val loss 0.2297 | val acc 0.9234
[q9_cosine_scores] epoch 4/5 | val loss 0.2151 | val acc 0.9274
[q9_cosine_scores] epoch 5/5 | val loss 0.2098 | val acc 0.9300
